# clock/01 — chunky condition-checking dataset (int | str)

Tests **only** the `CHUNKY_PREDICATES` bank: conditions whose implementation is inherently
multi-step (explicit loops / accumulators / flags — no one-line form). A spec is single-datatype
(`int` → `check_conditions(n: int)`, `str` → `check_conditions(s: str)`). The naming constraint
(`a`, `b`, `c`) is dropped and the instruction nudges toward multi-step implementations, so the
model doesn't collapse each check to a one-liner.

Pipeline: load chunky examples → model behavior (generate + eyeball + correctness peek) →
oracle-label the code segments (with the labelled-condition visualization) → gaze-head attention
on a few clean examples.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os


os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,6")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append("../..")


In [2]:
import logging

import torch
from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)

logging_utils.configure(level=logging.INFO)

In [3]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

2026-07-05 16:33:35 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models
2026-07-05 16:33:35 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-05 16:33:35 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json "HTTP/1.1 200 OK"
2026-07-05 16:33:35 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-05 16:33:35 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-05 16:33:35 ht

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-07-05 16:33:45 httpx INFO     HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-05 16:33:45 httpx INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json "HTTP/1.1 200 OK"
2026-07-05 16:33:46 src.models INFO     loaded model <Qwen/Qwen2.5-Coder-32B-Instruct> | size: 62492.135 MB | dtype: torch.bfloat16 | device: cuda:0
n_layer=64, n_embd=5120, n_heads=40


## 1. Load chunky dataset (int | str)

In [4]:
from src.goals.condition_data import sample_specs, materialize, chunky_predicates_for
from src.tokens import prepare_input

# ── config ──
N_SPECS         = 3               # per datatype
NUM_CONDITIONS  = [3, 4]
SEED            = 11
MAX_NEW_TOKENS  = 512
DATATYPES       = ["int", "str"]
# ────────────
chunky_samples = []
for dt in DATATYPES:
    dt_specs = sample_specs(N_SPECS, NUM_CONDITIONS, seed=SEED,
                            predicates=chunky_predicates_for(dt), allow_repeats=False)
    for j, sp in enumerate(dt_specs):
        sp.spec_id = f"chunky_{dt}_{j:02d}"
    chunky_samples += materialize(dt_specs, langs=["python"], modularities=["monolithic"])

print(f"{len(chunky_samples)} chunky samples "
      f"({sum(s.spec.datatype == 'int' for s in chunky_samples)} int, "
      f"{sum(s.spec.datatype == 'str' for s in chunky_samples)} str)")
for s in chunky_samples:
    print(f"  {s.spec.spec_id}  ({s.spec.num_conditions})  {s.spec.predicate_ids}")

6 chunky samples (3 int, 3 str)
  chunky_int_00  (4)  ['all_digits_even_chunky', 'digit_sum_gt_chunky', 'prime_chunky', 'perfect_number_chunky']
  chunky_int_01  (4)  ['digit_sum_gt_chunky', 'prime_chunky', 'perfect_number_chunky', 'all_digits_even_chunky']
  chunky_int_02  (3)  ['prime_chunky', 'all_digits_even_chunky', 'perfect_number_chunky']
  chunky_str_00  (4)  ['no_repeated_chars_chunky', 'palindrome_chunky', 'vowel_count_ge_chunky', 'has_digit_char_chunky']
  chunky_str_01  (4)  ['vowel_count_ge_chunky', 'no_repeated_chars_chunky', 'has_digit_char_chunky', 'palindrome_chunky']
  chunky_str_02  (4)  ['no_repeated_chars_chunky', 'vowel_count_ge_chunky', 'has_digit_char_chunky', 'palindrome_chunky']


In [5]:
from src.goals.functional import apply_chat_template, supports_chat_template

def build_prompt(sample):
    # name_rule="free" (no a/b/c constraint) + multistep nudge -> multi-line implementations
    instr = sample.render_instruction(name_rule="free", multistep=True)
    stub  = sample.render_code_stub()
    if supports_chat_template(mt):
        return apply_chat_template(mt=mt, user_query=instr.strip(), incomplete_code=stub)
    return instr + "\nAnswer:\n" + stub

# eyeball one prompt (a str-datatype sample)
print(build_prompt(next(s for s in chunky_samples if s.spec.datatype == "str")))

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a Python program that implements a function `check_conditions` which, given a string s, checks the following conditions:

1. no character in s appears more than once
2. s reads the same forwards and backwards
3. s contains at least 4 vowels
4. s contains at least one digit character

For each condition, in the order listed, assign the boolean result of the check to its own local variable. After all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each condition passed individually and whether all of them passed collectively.
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define any helper functions.
Implement each condition's check step by 

## 2. Model behavior — greedy generation

In [6]:
BATCH_SIZE = 8
_pad_id = mt.tokenizer.pad_token_id if mt.tokenizer.pad_token_id is not None else mt.tokenizer.eos_token_id

@torch.no_grad()
def batched_generate(samples, max_new=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
    outs = {}
    for i in range(0, len(samples), batch_size):
        chunk = samples[i:i + batch_size]
        inputs = prepare_input(prompts=[build_prompt(s) for s in chunk], tokenizer=mt)  # left-padded
        plen = inputs["input_ids"].shape[1]
        gen = mt._model.generate(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new, do_sample=False, pad_token_id=_pad_id,
        )
        for s, row in zip(chunk, gen[:, plen:]):  # only the new tokens; skip pad/eos on decode
            outs[(s.spec.spec_id, s.lang, s.modularity)] = mt.tokenizer.decode(row, skip_special_tokens=True)
    return outs


chunky_completions = batched_generate(chunky_samples)
print(f"generated {len(chunky_completions)} chunky completions")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


generated 6 chunky completions


In [7]:
# eyeball: stub + generation per chunky sample
for s in chunky_samples:
    out = chunky_completions[(s.spec.spec_id, s.lang, s.modularity)]
    rich_print(f"\n[bold cyan]== {s.spec.datatype} | {s.spec.spec_id}  "
               f"preds={s.spec.predicate_ids} ==[/bold cyan]")
    rich_print(f"[grey58]{escape(' | '.join(s.spec.spec_texts))}[/grey58]")
    rich_print(f"[salmon1]{escape(s.render_code_stub() + out)}[/salmon1]")

== int | chunky_int_00  preds=['all_digits_even_chunky', 'digit_sum_gt_chunky', 'prime_chunky', 
'perfect_number_chunky'] ==

all of n's digits are even | the sum of n's digits is greater than 20 | n is a prime number | the sum of n's proper
divisors equals n

def check_conditions(n: int):
    a = True
    for d in str(n):
        if int(d) % 2 != 0:
            a = False
            break
    b = False
    s = 0
    for d in str(n):
        s += int(d)
    if s > 20:
        b = True
    c = True
    if n < 2:
        c = False
    else:
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                c = False
                break
    d = False
    sd = 0
    for i in range(1, n):
        if n % i == 0:
            sd += i
    if sd == n:
        d = True
    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, Condition 4: {d}, All Conditions: {a and b and c 
and d}")
</code>

== int | chunky_int_01  preds=['digit_sum_gt_chunky', 'prime_chunky', 'perfect_number_chunky', 
'all_digits_even_chunky'] ==

the sum of n's digits is greater than 20 | n is a prime number | the sum of n's proper divisors equals n | all of 
n's digits are even

def check_conditions(n: int):
    sum_digits = 0
    temp = n
    while temp > 0:
        sum_digits += temp % 10
        temp //= 10
    condition1 = sum_digits > 20

    if n <= 1:
        condition2 = False
    else:
        condition2 = True
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                condition2 = False
                break

    sum_divisors = 0
    for i in range(1, n):
        if n % i == 0:
            sum_divisors += i
    condition3 = sum_divisors == n

    all_even = True
    temp = n
    while temp > 0:
        if temp % 10 % 2 != 0:
            all_even = False
            break
        temp //= 10
    condition4 = all_even

    print(f"Condition 1: {condition1}, Condition 2: {condition2}, Condition 3: {condition3}, Condition 4: 
{condition4}, All Conditions: {condition1 and condition2 and condition3 and condition4}")
</code>

... (8 outputs left out)

def check_conditions(s: str):
    vowels = 'aeiouAEIOU'
    vowel_count = 0
    char_count = {}
    digit_found = False
    is_palindrome = True

    for char in s:
        if char in vowels:
            vowel_count += 1
        if char in char_count:
            char_count[char] += 1
        else:
            char_count[char] = 1
        if char.isdigit():
            digit_found = True

    for i in range(len(s) // 2):
        if s[i] != s[-(i + 1)]:
            is_palindrome = False

    cond1 = vowel_count >= 2
    cond2 = all(count == 1 for count in char_count.values())
    cond3 = digit_found
    cond4 = is_palindrome

    print(f"Condition 1: {cond1}, Condition 2: {cond2}, Condition 3: {cond3}, Condition 4: {cond4}, All Conditions 
Passed: {cond1 and cond2 and cond3 and cond4}")
</code>

== str | chunky_str_02  preds=['no_repeated_chars_chunky', 'vowel_count_ge_chunky', 'has_digit_char_chunky', 
'palindrome_chunky'] ==

no character in s appears more than once | s contains at least 4 vowels | s contains at least one digit character |
s reads the same forwards and backwards

def check_conditions(s: str):
    unique_chars = True
    char_count = {}
    for char in s:
        if char in char_count:
            unique_chars = False
            break
        char_count[char] = 1

    vowel_count = 0
    vowels = 'aeiouAEIOU'
    for char in s:
        if char in vowels:
            vowel_count += 1
    has_four_vowels = vowel_count >= 4

    has_digit = False
    for char in s:
        if char.isdigit():
            has_digit = True
            break

    is_palindrome = True
    length = len(s)
    for i in range(length // 2):
        if s[i] != s[length - i - 1]:
            is_palindrome = False
            break

    print(f"Unique chars: {unique_chars}, Four vowels: {has_four_vowels}, Has digit: {has_digit}, Palindrome: 
{is_palindrome}, All conditions: {unique_chars and has_four_vowels and has_digit and is_palindrome}")
</code>

In [8]:
import io, contextlib

def peek_generated(sample, gen, n_show=2):
    """Exec the generated check_conditions and compare stdout to the oracle. Works for int AND
    str specs — uses the datatype-aware stub + the spec's own (int|str) test_inputs."""
    spec = sample.spec
    code = sample.render_code_stub() + gen
    ns = {}
    try:
        exec(code, ns)
    except Exception as e:
        rich_print(f"[red]exec failed: {e}[/red]"); return
    fn = ns.get("check_conditions")
    if fn is None:
        rich_print("[red]no check_conditions defined[/red]"); return
    for x in spec.test_inputs[:n_show]:
        buf = io.StringIO()
        try:
            with contextlib.redirect_stdout(buf): fn(x)
        except Exception as e:
            rich_print(f"[red]x={x!r}: runtime error {e}[/red]"); continue
        indiv, collective = spec.expected_results(x)
        rich_print(f"x={x!r}  expected individual={indiv}  all={collective}")
        rich_print(f"   stdout: {buf.getvalue().strip()!r}")

for s in chunky_samples:
    rich_print(f"\n[bold]-- {s.spec.datatype} | {s.spec.spec_id} --[/bold]")
    peek_generated(s, chunky_completions[(s.spec.spec_id, s.lang, s.modularity)], n_show=2)

-- int | chunky_int_00 --

exec failed: invalid syntax (<string>, line 29)

-- int | chunky_int_01 --

exec failed: invalid syntax (<string>, line 34)

-- int | chunky_int_02 --

exec failed: invalid syntax (<string>, line 18)

... (2 outputs left out)

-- str | chunky_str_01 --

exec failed: invalid syntax (<string>, line 28)

-- str | chunky_str_02 --

exec failed: invalid syntax (<string>, line 31)

## 3. Oracle labelling of the code segments

In [9]:
from src.goals.multistep_label import label_generation
from rich.text import Text
from rich.table import Table

PALETTE = ["green", "cyan", "magenta", "yellow", "blue", "bright_red",
           "bright_cyan", "bright_magenta", "bright_green"]
def _vis(s): return s.replace("\n", "↵\n").replace("\t", "⇥")

def render_oracle_labels(lab):
    combined = lab.full_prompt + lab.generation
    enc = mt.tokenizer(combined, return_offsets_mapping=True, add_special_tokens=False)
    ids = enc["input_ids"]; offs = [tuple(p) for p in enc["offset_mapping"]]
    char_off = len(lab.full_prompt)
    gen_start = next(j for j, (a, b) in enumerate(offs) if a >= char_off and b > a)
    prompt_step, gen_step, seg_start = {}, {}, {}
    for i, sl in enumerate(lab.labels):
        if sl.prompt_token_range:
            for t in range(*sl.prompt_token_range): prompt_step[t] = i
        if sl.gen_token_range:
            for t in range(*sl.gen_token_range): gen_step[t] = i
            seg_start.setdefault(sl.gen_token_range[0], i)
    sty = lambda i: f"black on {PALETTE[i % len(PALETTE)]}"
    rich_print(f"[bold]{lab.sample_name}[/bold]   (each condition = a color; spec span + code span share it)")
    txt = Text()
    for t in range(len(ids)):
        if t == gen_start:
            txt.append("\n" + "─" * 28 + " GENERATION " + "─" * 28 + "\n", style="bold grey42")
        if t in seg_start:
            txt.append(f"\n┌── cond {seg_start[t]} ──\n", style="bold yellow")
        s = _vis(mt.tokenizer.decode([ids[t]]))
        i = prompt_step.get(t) if t < gen_start else gen_step.get(t)
        base = "grey42" if t < gen_start else "grey58"
        txt.append(s, style=(sty(i) if i is not None else base))
    console.print(txt)
    tb = Table(title="oracle step labels", show_lines=True)
    for col, w in [("cond", 4), ("condition", 34), ("prompt tok", 12),
                   ("gen lines", 10), ("gen tok", 12), ("gen_code", 40), ("reasoning", 36)]:
        tb.add_column(col, overflow="fold", max_width=w)
    for i, sl in enumerate(lab.labels):
        tb.add_row(str(i), sl.step_text, str(sl.prompt_token_range), str(sl.gen_line_range),
                   str(sl.gen_token_range), _vis(sl.gen_code), sl.reasoning)
    console.print(tb)


In [10]:
# oracle-label one str + one int chunky example and show the labelled condition/code segments
for _dt in ("str", "int"):
    _c  = next(s for s in chunky_samples if s.spec.datatype == _dt)
    _cp = build_prompt(_c)
    _cg = chunky_completions[(_c.spec.spec_id, _c.lang, _c.modularity)]
    _lab = label_generation(mt, _c, _cp, _cg, use_cache=True)
    print(f"labeled {_lab.sample_name}  ({len(_lab.labels)} conditions)")
    render_oracle_labels(_lab)

2026-07-05 16:34:10 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
labeled chunky_str_00__python__monolithic  (4 conditions)


chunky_str_00__python__monolithic   (each condition = a color; spec span + code span share it)

<|im_start|>system↵
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>↵
<|im_start|>user↵
Write a Python program that implements a function `check_conditions` which, given a string s, checks the following 
conditions:↵
↵
1. no character in s appears more than once↵
2. s reads the same forwards and backwards↵
3. s contains at least 4 vowels↵
4. s contains at least one digit character↵
↵
For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.↵
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.↵
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.↵
Do not include any comments in your code.<|im_end|>↵
<|im_start|>assistant↵
<code>↵
def check_conditions(s: str):↵

──────────────────────────── GENERATION ────────────────────────────

┌── cond 0 ──
    unique_chars = True↵
    seen_chars = set()↵
    for char in s:↵
        if char in seen_chars:↵
            unique_chars = False↵
        seen_chars.add(char)↵
↵

┌── cond 1 ──
    palindrome = True↵
    n = len(s)↵
    for i in range(n // 2):↵
        if s[i] != s[n - i - 1]:↵
            palindrome = False↵
↵

┌── cond 2 ──
    vowel_count = 0↵
    vowels = "aeiouAEIOU"↵
    for char in s:↵
        if char in vowels:↵
            vowel_count += 1↵
    enough_vowels = vowel_count >= 4↵
↵

┌── cond 3 ──
    digit_present = False↵
    for char in s:↵
        if char.isdigit():↵
            digit_present = True↵
↵
    print(f"Unique chars: {unique_chars}, Palindrome: {palindrome}, Enough vowels: {enough_vowels}, Digit present: 
{digit_present}, All conditions met: {unique_chars and palindrome and enough_vowels and digit_present}")↵
</code>

                                                oracle step labels                                                 
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ cond ┃ condition            ┃ prompt tok ┃ gen lines ┃ gen tok    ┃ gen_code             ┃ reasoning            ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ 0    │ no character in s    │ (70, 78)   │ (1, 6)    │ (241, 277) │     unique_chars =   │ Lines 1-6 check for  │
│      │ appears more than    │            │           │            │ True↵                │ unique characters by │
│      │ once                 │            │           │            │     seen_chars =     │ tracking seen        │
│      │                      │            │           │            │ set()↵               │ characters in a set  │
│      │                      │            │           │            │     for char in s:↵  │ and setting          │
│      │                      │            │           │            │         if char in   │ unique_chars to      │
│      │                      │            │           │            │ seen_chars:↵         │ False if any         │
│      │                      │            │           │            │             unique_c │ duplicate is found.  │
│      │                      │            │           │            │ hars = False↵        │                      │
│      │                      │            │           │            │         seen_chars.a │                      │
│      │                      │            │           │            │ dd(char)↵            │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 1    │ s reads the same     │ (81, 88)   │ (8, 12)   │ (278, 317) │     palindrome =     │ Lines 8-12 check if  │
│      │ forwards and         │            │           │            │ True↵                │ the string is a      │
│      │ backwards            │            │           │            │     n = len(s)↵      │ palindrome by        │
│      │                      │            │           │            │     for i in range(n │ comparing characters │
│      │                      │            │           │            │ // 2):↵              │ from both ends       │
│      │                      │            │           │            │         if s != s:↵  │ toward the middle.   │
│      │                      │            │           │            │             palindro │                      │
│      │                      │            │           │            │ me = False↵          │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 2    │ s contains at least  │ (91, 98)   │ (14, 19)  │ (318, 365) │     vowel_count = 0↵ │ Lines 14-19 count    │
│      │ 4 vowels             │            │           │            │     vowels =         │ vowels in the string │
│      │                      │            │           │            │ "aeiouAEIOU"↵        │ and check if the     │
│      │                      │            │           │            │     for char in s:↵  │ count is at least 4. │
│      │                      │            │           │            │         if char in   │                      │
│      │                      │            │           │            │ vowels:↵             │                      │
│      │                      │            │           │            │             vowel_co │                      │
│      │                      │            │           │

2026-07-05 16:34:14 httpx INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
labeled chunky_int_00__python__monolithic  (4 conditions)


chunky_int_00__python__monolithic   (each condition = a color; spec span + code span share it)

<|im_start|>system↵
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>↵
<|im_start|>user↵
Write a Python program that implements a function `check_conditions` which, given an integer n, checks the 
following conditions:↵
↵
1. all of n's digits are even↵
2. the sum of n's digits is greater than 20↵
3. n is a prime number↵
4. the sum of n's proper divisors equals n↵
↵
For each condition, in the order listed, assign the boolean result of the check to its own local variable. After 
all conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each 
condition passed individually and whether all of them passed collectively.↵
Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not define
any helper functions.↵
Implement each condition's check step by step using explicit statements — loops and intermediate variables where 
appropriate — rather than a single one-line expression or comprehension.↵
Do not include any comments in your code.<|im_end|>↵
<|im_start|>assistant↵
<code>↵
def check_conditions(n: int):↵

──────────────────────────── GENERATION ────────────────────────────

┌── cond 0 ──
    a = True↵
    for d in str(n):↵
        if int(d) % 2 != 0:↵
            a = False↵
            break↵

┌── cond 1 ──
    b = False↵
    s = 0↵
    for d in str(n):↵
        s += int(d)↵
    if s > 20:↵
        b = True↵

┌── cond 2 ──
    c = True↵
    if n < 2:↵
        c = False↵
    else:↵
        for i in range(2, int(n**0.5) + 1):↵
            if n % i == 0:↵
                c = False↵
                break↵

┌── cond 3 ──
    d = False↵
    sd = 0↵
    for i in range(1, n):↵
        if n % i == 0:↵
            sd += i↵
    if sd == n:↵
        d = True↵
    print(f"Condition 1: {a}, Condition 2: {b}, Condition 3: {c}, Condition 4: {d}, All Conditions: {a and b and c 
and d}")↵
</code>

                                                oracle step labels                                                 
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ cond ┃ condition            ┃ prompt tok ┃ gen lines ┃ gen tok    ┃ gen_code             ┃ reasoning            ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ 0    │ all of n's digits    │ (70, 77)   │ (1, 5)    │ (246, 278) │     a = True↵        │ Lines 1-5 initialize │
│      │ are even             │            │           │            │     for d in         │ `a` to True and      │
│      │                      │            │           │            │ str(n):↵             │ check if any digit   │
│      │                      │            │           │            │         if int(d) %  │ of n is odd, setting │
│      │                      │            │           │            │ 2 != 0:↵             │ `a` to False if so.  │
│      │                      │            │           │            │             a =      │                      │
│      │                      │            │           │            │ False↵               │                      │
│      │                      │            │           │            │             break↵   │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 1    │ the sum of n's       │ (80, 92)   │ (6, 11)   │ (278, 315) │     b = False↵       │ Lines 6-11           │
│      │ digits is greater    │            │           │            │     s = 0↵           │ initialize `b` to    │
│      │ than 20              │            │           │            │     for d in         │ False, sum the       │
│      │                      │            │           │            │ str(n):↵             │ digits of n, and set │
│      │                      │            │           │            │         s += int(d)↵ │ `b` to True if the   │
│      │                      │            │           │            │     if s > 20:↵      │ sum exceeds 20.      │
│      │                      │            │           │            │         b = True↵    │                      │
│      │                      │            │           │            │                      │                      │
├──────┼──────────────────────┼────────────┼───────────┼────────────┼──────────────────────┼──────────────────────┤
│ 2    │ n is a prime number  │ (95, 100)  │ (12, 19)  │ (315, 371) │     c = True↵        │ Lines 12-19          │
│      │                      │            │           │            │     if n < 2:↵       │ initialize `c` to    │
│      │                      │            │           │            │         c = False↵   │ True and check       │
│      │                      │            │           │            │     else:↵           │ primality by testing │
│      │                      │            │           │            │         for i in     │ divisibility up to   │
│      │                      │            │           │            │ range(2, int(n**0.5) │ the square root of   │
│      │                      │            │           │            │ + 1):↵               │ n.                   │
│      │                      │            │           │            │             if n % i │                      │
│      │                      │            │           │            │ == 0:↵               │                      │
│      │                      │            │           │            │                 c =  │                      │
│      │                      │            │           │            │ False↵               │                      │
│      │                      │            │           │

## 4. Gaze heads + attention on clean examples

In [11]:
from scripts.clock.score_condition_heads import load_head_scores
import numpy as np

# ── pick the scoring config to inspect ────────────────────────────────────────
Q_SPAN = 15
COMPARE_WITH = "mean"        # "mean" | "max"
TOP_K = 100
VERSION = "conditions"
# ──────────────────────────────────────────────────────────────────────────────
gaze_scores = load_head_scores(model=MODEL_NAME, version=VERSION, q_span=Q_SPAN, compare_with=COMPARE_WITH)["combined"]

# ── top-K gaze heads ──────────────────────────────────────────────────────────
flat_idx = np.argsort(gaze_scores, axis=None)[::-1][:TOP_K]
gaze_heads = [(int(l), int(h)) for l, h in (np.unravel_index(i, gaze_scores.shape) for i in flat_idx)]
print(f"top-{TOP_K} gaze heads  (compare_with='{COMPARE_WITH}', q_span={Q_SPAN})  "
      f"score range [{gaze_scores[gaze_heads[-1]]:.3f}, {gaze_scores[gaze_heads[0]]:.3f}]")
for rank, (l, h) in enumerate(gaze_heads[:15], start=1):
    print(f"  {rank:2d}. L{l:2d} H{h:2d}  score={gaze_scores[l, h]:.3f}")
if TOP_K > 15:
    print(f"  ... (+ {TOP_K - 15} more)")

top-100 gaze heads  (compare_with='mean', q_span=15)  score range [2.725, 3.478]
   1. L53 H16  score=3.478
   2. L52 H23  score=3.464
   3. L40 H21  score=3.452
   4. L51 H13  score=3.451
   5. L36 H23  score=3.422
   6. L59 H34  score=3.364
   7. L34 H11  score=3.358
   8. L45 H12  score=3.293
   9. L48 H 8  score=3.277
  10. L51 H11  score=3.246
  11. L45 H11  score=3.222
  12. L48 H 5  score=3.217
  13. L50 H12  score=3.181
  14. L32 H23  score=3.175
  15. L53 H18  score=3.168
  ... (+ 85 more)


In [12]:
from src.attention import get_attention_matrices, show_interactive_attention

# gaze-head attention on a clean chunky generation (one per datatype)
for _dt in ("int", "str"):
    _ca  = next(s for s in chunky_samples if s.spec.datatype == _dt)
    _cag = chunky_completions[(_ca.spec.spec_id, _ca.lang, _ca.modularity)]
    clean_full = (build_prompt(_ca) + _cag).replace("\n", "    \n")  # uniform newline tokenization
    attn_info = get_attention_matrices(input=clean_full, mt=mt, value_weighted=True)
    show_interactive_attention(
        attn_info=attn_info,
        selected_heads=sorted(gaze_heads),
        start_from=0, agg="mean", normalize="row_max", cmap="Blues",
        title=f"gaze heads on chunky {_dt} — {_ca.spec.spec_id}",
        track_conditions=_ca.spec,      # ConditionSpec -> per-condition gaze tooltips
        height=None,
    )